# 03 — Composite Validation (multi-cycle, DCC process **V1**)
**SEA-FORWARD** · OPERA Capacity Development · OceanPrediction-A toolkit

`02_validation.ipynb` validates ONE forecast cycle, day by calendar date.
This notebook instead **merges several cycles** into one composite
analysis, indexed by **forecast lead time** rather than calendar date:

    cycle 20260701's days -> sp1, sp2, fcst1, fcst2, fcst3, ...
    cycle 20260706's days -> sp1, sp2, fcst1, fcst2, fcst3, ...
    cycle 20260711's days -> sp1, sp2, fcst1, fcst2, fcst3, fcst4
        |
        v  merged/pooled BY LABEL, not by calendar date
    sp1, sp2 : model spin-up (excluded from pass/fail, same convention as
               02_validation.ipynb Section 6/SPINUP_DAYS)
    fcst1, fcst2, fcst3, ... : genuine forecast lead days, POOLED across
               every cycle that reaches that lead

so "how skillful is the forecast on its 3rd day" can be assessed across
every cycle at once, instead of one cycle's absolute dates. Copernicus
Marine and satellite reference data are downloaded/checked per cycle
(same logic as 02_validation.ipynb Section 1c) and merged the same way --
by lead label, not calendar date.

Everything here is a thin wrapper around `sftools.validation_composite`
(`vc`), which itself is a thin per-lead ACCUMULATOR around the same
`sftools.validation` (`val`) building blocks 02_validation.ipynb uses --
so a composite run and a single-cycle run can never silently disagree on
methodology.


In [2]:
# ----------------------------------------------------------------------
# Setup -- run from the notebooks/ folder so sftools imports (see sftools/README.md)
# ----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, "..")   # repo root, so `import sftools...` resolves

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import glob
import uuid

import sftools.postprocess as pp
import sftools.validation as val               # single-cycle building blocks (maps,
                                               # scatter, GODAE metrics, satellite,
                                               # HTML summary, ...)
import sftools.validation_composite as vc      # composite (multi-cycle, lead-time) layer
import sftools.plotting as pl

import _paths


## 0. Cycles to composite

Every forecast run lives in a cycle directory named `YYYYMMDD` (ex: ```20260729```), sibling
to every other cycle under `<MAIN_DIR>/<CONFIG>/` (see
`notebooks/_paths.py`). Pick **at least 2** cycles below to merge into one
composite -- they don't need to be the same length (a 5-day and a 7-day
forecast cycle can be composited together; the 7-day one just contributes
to `fcst6`/`fcst7` on its own, see Section 1c).


In [3]:
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
MAIN_DIR = os.path.expanduser(
    os.environ.get("SEAFORWARD_MAIN_DIR", "~/seaforward/forecast/model-runs"))

AVAILABLE_CYCLES = _paths.list_cycles(MAIN_DIR, CONFIG)
print(f"Forecast cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")


Forecast cycles found under /home/lell/seaforward/forecast/model-runs/Canary_12: ['20260711', '20260723', '20260729']


In [5]:
# >>> SET THIS to the cycles you want to composite, e.g. at least 2 <
# Leave the env var unset AND the list below empty for EVERY available cycle
# under MAIN_DIR/CONFIG to be selected (AVAILABLE_CYCLES above).
CYCLES = os.environ.get("SEAFORWARD_CYCLES", "").split(",") if os.environ.get("SEAFORWARD_CYCLES") \
        else []   # <<<< write here at least two cycles you wish to validate
if not CYCLES:
    CYCLES = list(AVAILABLE_CYCLES)

if len(CYCLES) < 2:
    raise ValueError(f"CYCLES must have at least 2 entries for a composite validation, got {CYCLES!r}")

YORIG = 2000   # if not working, set to >>> None <<< since real CROCO output carries proper CF time units

# >>> Depth level for the grid comparisons in Section 2 (SST/currents/SSS maps) <
# None -> surface. SSH has no depth dimension, always surface regardless of this.
DEPTH_M = None

# The first SPINUP_DAYS day(s) of EVERY cycle carry CROCO's model start-up
# transient -- labelled 'sp1','sp2',... (kept separate from 'fcst1','fcst2',...)
# so they're visible in the boxplots (Section 5b/5c) but excluded from the
# pass/fail summary (Section 5), exactly like 02_validation.ipynb Section 6.
SPINUP_DAYS = 2

# coordinates for point timeseries and vertical profiles
POINT_LON, POINT_LAT = -17.0, 18.0   # <<<< set to your point of interest


```COMPOSITE_DIR``` is named with a short uuid rather than the cycle list itself
 (```CYCLES``` can be long / list every cycle on disk, which would make an
 unreadably long directory name) -- the actual cycle list this run
 composited is written to ```validated_cycles.txt``` inside that directory.



The **uuid** is only minted once per distinct set of cycles: before creating a
 new one, every existing ```validation_composite_*``` folder under
 ```MAIN_DIR/CONFIG``` is checked, and if one already has a ```validated_cycles.txt```
 listing exactly this same set of cycles (order doesn't matter), that
 folder is reused -- so re-running this notebook on the same CYCLES keeps
 writing into the same ```COMPOSITE_DIR``` instead of a new random one every
 time. 
 
 **Set SEAFORWARD_COMPOSITE_ID yourself to override this and force a specific id (e.g. to deliberately start a fresh composite for the same cycles)**.
 - replace ```os.environ.get("SEAFORWARD_COMPOSITE_ID")``` with your desired 8-character id, ex: ```"fg03sc40"```

In [6]:
FORCE_COMPOSITE_ID = os.environ.get("SEAFORWARD_COMPOSITE_ID")   # <<<< set to force a specific/new id
COMPOSITE_ID, COMPOSITE_DIR = vc.resolve_composite_dir(MAIN_DIR, CONFIG, CYCLES, force_id=FORCE_COMPOSITE_ID)


Reusing existing composite directory for this exact cycle set: /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6
Compositing 3 cycle(s): ['20260711', '20260723', '20260729']
Composite outputs -> /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6
  (cycle list also recorded in /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6/validated_cycles.txt)


## 1b. Reference-product availability

Same check as 02_validation.ipynb Section 1b -- shared across every
cycle in this composite, since it's a platform-level (not per-cycle)
availability check.


In [7]:
AVAIL = {
    name: val.dataset_available(name)
    for name in ("mercator_forecast", "ostia_l4", "odyssea_l3s", "smos_l4_sss")
}

print()
print(AVAIL)


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.57s/it]

  CMEMS product 'mercator_forecast' (cmems_mod_glo_phy_anfc_0.083deg_P1D-m): available



Fetching catalogue 1: 100%|███████████████████████████████████████████████████████████████| 2/2 [00:04<00:00,  2.30s/it]
                                                                                                                        
Fetching catalogue 1: 100%|███████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.58s/it]


  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.18s/it]

  CMEMS product 'odyssea_l3s' (IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE): available



Fetching catalogue 1:   0%|                                                                       | 0/2 [00:00<?, ?it/s]

Fetching catalogue 1: 100%|███████████████████████████████████████████████████████████████| 2/2 [00:07<00:00,  3.80s/it]


Fetching products: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.51it/s]

                                                                                                                        
Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:02<00:02,  2.98s/it]

  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available

{'mercator_forecast': True, 'ostia_l4': True, 'odyssea_l3s': True, 'smos_l4_sss': True}


## 1c. Resolve paths, download reference data, and label each cycle's
## days by forecast lead time

For each cycle: resolve `CROCO_HIS`/`REFERENCE`, download (or reuse
already-downloaded) the Copernicus Marine Forecast + satellite SST/SSS --
IDENTICAL per-cycle logic to 02_validation.ipynb Section 1c, just looped
over `CYCLES` -- then assign every calendar day in that cycle a
**lead-time label** (`vc.label_cycle_days`, see the intro above).
`cycles_info` (a list of one dict per cycle) is what every composite
function below takes as its first argument.

Numerical-stability check (NaN/Inf in the CROCO output) is done per cycle
here too, matching 02_validation.ipynb Section 1's `stability_ok`.


In [8]:
cycles_info, LEADS, FCST_LEADS, stability_ok = vc.download_and_label_cycles(
    CYCLES, CONFIG, MAIN_DIR, AVAIL, YORIG, SPINUP_DAYS, _paths)



== cycle 20260711 ==
  Mercator: already downloaded and covers the full cycle window: /home/lell/seaforward/forecast/model-runs/Canary_12/20260711/downloaded_data/MERCATOR/MERCATOR_20260711_00.nc


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.81s/it]INFO - 2026-09-13T12:50:06Z - Checking if credentials are valid.


  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available


INFO - 2026-09-13T12:50:08Z - Valid credentials from configuration file.


CMEMS: already logged in.
  [OSTIA] 2026-07-11: already downloaded - 2026-07-11.nc
  [OSTIA] 2026-07-12: already downloaded - 2026-07-12.nc
  [OSTIA] 2026-07-13: already downloaded - 2026-07-13.nc
  [OSTIA] 2026-07-14: already downloaded - 2026-07-14.nc
  [OSTIA] 2026-07-15: already downloaded - 2026-07-15.nc
  [OSTIA] 2026-07-16: already downloaded - 2026-07-16.nc


...

CMEMS: already logged in.
  [ODYSSEA] 2026-07-28: already downloaded - 2026-07-28.nc
  [ODYSSEA] 2026-07-29: already downloaded - 2026-07-29.nc
  [ODYSSEA] 2026-07-30: already downloaded - 2026-07-30.nc
  [ODYSSEA] 2026-07-31: already downloaded - 2026-07-31.nc
  [ODYSSEA] 2026-08-01: already downloaded - 2026-08-01.nc
  [ODYSSEA] 2026-08-02: already downloaded - 2026-08-02.nc
  [ODYSSEA] 2026-08-03: already downloaded - 2026-08-03.nc
  [ODYSSEA] 2026-08-04: already downloaded - 2026-08-04.nc


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.29s/it]INFO - 2026-09-13T12:51:29Z - Checking if credentials are valid.


  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available


INFO - 2026-09-13T12:51:30Z - Valid credentials from configuration file.


CMEMS: already logged in.
  [SMOS] 2026-07-28: already downloaded - 2026-07-28.nc
  [SMOS] 2026-07-29: already downloaded - 2026-07-29.nc
  [SMOS] 2026-07-30: already downloaded - 2026-07-30.nc
  [SMOS] 2026-07-31: already downloaded - 2026-07-31.nc
  [SMOS] 2026-08-01: already downloaded - 2026-08-01.nc
  [SMOS] 2026-08-02: already downloaded - 2026-08-02.nc
  [SMOS] 2026-08-03: already downloaded - 2026-08-03.nc
  [SMOS] 2026-08-04: already downloaded - 2026-08-04.nc

  days: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']
  lead labels: {'2026-07-28': 'sp1', '2026-07-29': 'sp2', '2026-07-30': 'fcst1', '2026-07-31': 'fcst2', '2026-08-01': 'fcst3', '2026-08-02': 'fcst4', '2026-08-03': 'fcst5', '2026-08-04': 'fcst6'}

All lead labels in this composite : ['sp1', 'sp2', 'fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']
Forecast-only lead labels (Section 5): ['fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']
Numerica

## 2. Composite bias maps -- CROCO forecast vs Copernicus Marine
## Forecast, by lead time

Composite counterpart to 02_validation.ipynb Section 2: instead of one
2x2 figure per CALENDAR DAY of a single cycle, ONE 2x2 figure per LEAD
TIME (`sp1`, `sp2`, `fcst1`, `fcst2`, ...), composited (pixel-averaged)
across every cycle in `cycles_info` that reaches that lead. A given
lead's figure shows the composite-mean CROCO field, composite-mean
Copernicus field, the mean bias across contributing cycles, and the
RMSE across contributing cycles (same accumulation math as
`sftools.validation._multiday_bias_rmse`, just one sample per
CONTRIBUTING CYCLE instead of one per day within a single cycle).

`sst_stats_by_lead`/`ssh_stats_by_lead`/`salt_stats_by_lead` mirror
02_validation.ipynb's `*_stats_by_day` dicts, keyed by lead instead of date.


In [9]:
sst_stats_by_lead = vc.validate_sst_maps(cycles_info, LEADS, YORIG, DEPTH_M, COMPOSITE_DIR,
                                        AVAIL['mercator_forecast'])


temperature @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.116  RMSE=0.225  cRMSE=0.192  corr=0.997
temperature @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.194  RMSE=0.320  cRMSE=0.255  corr=0.995
temperature @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.197  RMSE=0.344  cRMSE=0.282  corr=0.993
temperature @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.352  RMSE=0.470  cRMSE=0.311  corr=0.990
temperature @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.431  RMSE=0.536  cRMSE=0.318  corr=0.988
temperature @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [temperature]  n=8209  bias=-0.541  RMSE=0.635  cRMSE=0.332  corr=0.986
temperature @ fcst5  (composite of 1 cycle(s

In [10]:
ssh_stats_by_lead = vc.validate_ssh_maps(cycles_info, LEADS, YORIG, COMPOSITE_DIR,
                                        AVAIL['mercator_forecast'])


SSH anomaly @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.006  cRMSE=0.006  corr=0.995
SSH anomaly @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.007  cRMSE=0.007  corr=0.992
SSH anomaly @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.008  cRMSE=0.008  corr=0.990
SSH anomaly @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.009  cRMSE=0.009  corr=0.987
SSH anomaly @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.011  cRMSE=0.011  corr=0.982
SSH anomaly @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.013  cRMSE=0.013  corr=0.976
SSH anomaly @ fcst5  (composite of 1 cycle(s

In [11]:
salt_stats_by_lead = vc.validate_sss_maps(cycles_info, LEADS, YORIG, DEPTH_M, COMPOSITE_DIR,
                                         AVAIL['mercator_forecast'])


salinity @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.013  RMSE=0.059  cRMSE=0.058  corr=0.989
salinity @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.017  RMSE=0.076  cRMSE=0.074  corr=0.981
salinity @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.022  RMSE=0.092  cRMSE=0.089  corr=0.971
salinity @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.030  RMSE=0.115  cRMSE=0.111  corr=0.960
salinity @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.033  RMSE=0.130  cRMSE=0.126  corr=0.948
salinity @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [salinity]  n=8209  bias=+0.036  RMSE=0.140  cRMSE=0.136  corr=0.940
salinity @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [salinity]  n=8209 

In [12]:
cur_stats_by_lead = vc.validate_current_maps(cycles_info, LEADS, YORIG, DEPTH_M, COMPOSITE_DIR,
                                            AVAIL['mercator_forecast'])


speed @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.008  RMSE=0.047  cRMSE=0.046  corr=0.828
speed @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.010  RMSE=0.039  cRMSE=0.037  corr=0.859
speed @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.024  RMSE=0.050  cRMSE=0.044  corr=0.850
speed @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=+0.001  RMSE=0.048  cRMSE=0.048  corr=0.792
speed @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=-0.015  RMSE=0.053  cRMSE=0.051  corr=0.764
speed @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [speed]  n=8209  bias=-0.014  RMSE=0.063  cRMSE=0.061  corr=0.603
speed @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [speed]  n=8209  bias=-0.007  RMSE=0.091  cRMSE=0.091  cor

## 3. Composite scatter plots -- pointwise CROCO vs Copernicus Marine
## Forecast, by lead time

Composite counterpart to 02_validation.ipynb Section 3: one SST+SSH
scatter figure per lead time, with points POOLED (concatenated) from
every cycle that reaches that lead -- so, e.g., `fcst2`'s scatter
combines every contributing cycle's 3rd-day points into one distribution
and one bias/RMSE/corr annotation, rather than scoring each cycle
separately.


In [13]:
scatter_figs = vc.scatter_maps_by_lead(cycles_info, LEADS, YORIG, COMPOSITE_DIR,
                                       AVAIL['mercator_forecast'])


-> 8 figure(s) written, one per lead: ['sp1', 'sp2', 'fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']


## 4. Composite GODAE scorecard & Taylor diagram, by lead time

Composite counterpart to 02_validation.ipynb Section 4: for each lead
time, the GODAE scorecard (`bias`/`RMSD`/`uRMSD`/`corr`/`SI`/`std_ratio`)
for every variable (`temp`, `ssh`, `salt`, `speed`) is computed on points
POOLED across every contributing cycle (`vc.godae_scorecard_composite`),
then one Taylor diagram is drawn per lead, summarising all four
variables' pooled skill at that lead time.


In [14]:
report, report_by_lead = vc.build_godae_scorecard(cycles_info, LEADS, YORIG, DEPTH_M,
                                                   AVAIL['mercator_forecast'])


-- sp1  (cycles: ['20260711', '20260723', '20260729']) --
variable        vs layer     n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 24627  0.013 0.291  0.291 0.992  1.158  12.955
     ssh reference   all 24627 -0.000 0.009  0.009 0.985 23.268  17.204
    salt reference   all 24627  0.016 0.082  0.081 0.978  0.224  21.428
   speed reference   all 24627  0.045 0.110  0.101 0.535 59.838 100.270
-- sp2  (cycles: ['20260711', '20260723', '20260729']) --
variable        vs layer     n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 24627 -0.046 0.331  0.327 0.990  1.300  14.585
     ssh reference   all 24627 -0.000 0.010  0.010 0.983 24.685  18.355
    salt reference   all 24627  0.020 0.098  0.096 0.968  0.265  25.683
   speed reference   all 24627  0.058 0.110  0.094 0.624 53.650 100.284
-- fcst1  (cycles: ['20260711', '20260723', '20260729']) --
variable        vs layer     n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 24627 -

In [15]:
taylor_figs = vc.plot_taylor_diagrams_by_lead(report_by_lead, COMPOSITE_DIR)


-> 8 figure(s) written, one per lead: ['sp1', 'sp2', 'fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']


## 5. Automated pass/fail summary (V1) 
**-- composite, forecast leads only**

Composite counterpart to 02_validation.ipynb Section 5, with one
simplification: since spin-up is already a SEPARATE set of lead labels
(`sp1`, `sp2`, ...) rather than the first N calendar days of one cycle,
excluding it is just `report[report['lead'].isin(FCST_LEADS)]` -- no
day-counting/slicing needed. Each criterion is scored against the mean
across every (cycle, fcst-lead) pair pooled in `FCST_LEADS`.


In [16]:
all_pass = vc.pass_fail_summary(report, CYCLES, FCST_LEADS, stability_ok, spinup_days=SPINUP_DAYS)


Composite scorecard: 3 cycle(s) ['20260711', '20260723', '20260729'], forecast leads only (spin-up ['sp1', 'sp2'] excluded): ['fcst1', 'fcst2', 'fcst3', 'fcst4', 'fcst5', 'fcst6']

Criterion                                          Result  Value
---------------------------------------------------------------------------
SST composite-mean domain-avg RMSD < 0.5 degC      FAIL    0.569 degC
SSH composite-mean spatial correlation > 0.90      PASS    0.953
Salinity composite-mean domain-avg |bias| < 0.2 PSU PASS    +0.036 PSU
Numerical stability (no NaN/Inf, every cycle)      PASS    see section 1c
---------------------------------------------------------------------------


Worst single lead -- SST RMSD:   fcst6  (0.864 degC)
Worst single lead -- SSH corr:   fcst6  (0.914)
Worst single lead -- Salt bias:  fcst5  (+0.045 PSU)

Note: surface-current skill (by_var['speed']) has no fixed Section 9.3
threshold -- its criterion is qualitative (inspect the vector maps in
Section 2), same as 02_v

### 5b. Composite domain-wide bias boxplot, by lead time

Composite counterpart to 02_validation.ipynb Section 5b: one box per
LEAD TIME (spin-up included, so the transient is visible), pooling
`vc.composite_domain_diff` across every contributing cycle at that lead --
plotted with the exact same `sftools.validation.bias_boxplot` used by the
single-cycle notebook (no separate composite plotting code needed).


In [17]:
vc.stacked_bias_boxplot(cycles_info, LEADS, YORIG, DEPTH_M, COMPOSITE_DIR, CYCLES,
                       AVAIL['mercator_forecast'])


### 5c. Composite CROCO-vs-satellite domain-wide bias boxplot, by lead time

Composite counterpart to 02_validation.ipynb Section 5c/6c: OSTIA and
ODYSSEA grouped side by side, SMOS on its own boxplot, one box-group per
LEAD TIME, pooling `vc.composite_domain_diff_satellite` across every
contributing cycle -- plotted with `sftools.validation.bias_boxplot_multi`,
same as the single-cycle notebook.


In [18]:
any_sat_avail = vc.satellite_bias_boxplots(cycles_info, AVAIL, LEADS, YORIG, COMPOSITE_DIR, CYCLES)


### 5d. Composite point-wide mean timeseries, by lead time

Composite mean timeseries (CROCO vs parent, CROCO vs satellite)

Point + full-domain mean +/- 1 std (fill_between) per lead, for CROCO vs
Copernicus Marine Forecast and CROCO vs satellite -- set `POINT_LON`/
`POINT_LAT` below.

In [19]:
vc.plot_composite_timeseries(cycles_info, LEADS, POINT_LON, POINT_LAT, DEPTH_M, YORIG,
                             COMPOSITE_DIR, CYCLES, AVAIL['mercator_forecast'], any_sat_avail)


### 5e. Composite vertical profiles (point and full-domain), temp/salt/speed

Mean +/- 1 std (fill_between) across every cycle reaching `PROFILE_LEAD`,
CROCO vs parent.

In [20]:
vc.composite_profiles(cycles_info, LEADS, POINT_LON, POINT_LAT, YORIG, COMPOSITE_DIR)


## 6. Composite satellite SST validation map -- OSTIA & ODYSSEA, by lead time

Composite counterpart to 02_validation.ipynb Section 7: one 2x2 figure
per lead time (CROCO composite mean | satellite composite mean / bias |
RMSE), composited across every cycle with a downloaded file for that
lead's day.


In [21]:
sat_avail_key = {"OSTIA": "ostia_l4", "ODYSSEA": "odyssea_l3s", "SMOS": "smos_l4_sss"}
sat_stats_all = {}
for product in ("OSTIA", "ODYSSEA"):
    stats = vc.validate_satellite(cycles_info, product, AVAIL[sat_avail_key[product]], LEADS,
                                   YORIG, COMPOSITE_DIR)
    if stats is not None:
        sat_stats_all[product] = stats



-- OSTIA --
[OSTIA] SST @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=+0.186  RMSE=0.446  cRMSE=0.405  corr=0.981
[OSTIA] SST @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=+0.163  RMSE=0.446  cRMSE=0.415  corr=0.980
[OSTIA] SST @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=+0.191  RMSE=0.465  cRMSE=0.424  corr=0.978
[OSTIA] SST @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=-0.053  RMSE=0.465  cRMSE=0.462  corr=0.976
[OSTIA] SST @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=-0.063  RMSE=0.444  cRMSE=0.439  corr=0.978
[OSTIA] SST @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SST]  n=8257  bias=-0.089  RMSE=0.499  cRMSE=0.491  corr=0.973
[OSTIA] SST @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [SST]  n=8257  

## 6b. Composite satellite SSS validation -- SMOS, by lead time

Composite counterpart to 02_validation.ipynb Section 7b -- same pattern
as Section 6 above, for SMOS SSS instead of OSTIA/ODYSSEA SST.


In [22]:
smos_stats = vc.validate_satellite(cycles_info, "SMOS", AVAIL['smos_l4_sss'], LEADS,
                                   YORIG, COMPOSITE_DIR)
if smos_stats is not None:
    sat_stats_all["SMOS"] = smos_stats



-- SMOS --
[SMOS] SSS @ sp1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.091  RMSE=0.263  cRMSE=0.247  corr=0.664
[SMOS] SSS @ sp2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.085  RMSE=0.259  cRMSE=0.245  corr=0.638
[SMOS] SSS @ fcst1  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.077  RMSE=0.272  cRMSE=0.261  corr=0.562
[SMOS] SSS @ fcst2  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.082  RMSE=0.290  cRMSE=0.278  corr=0.463
[SMOS] SSS @ fcst3  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.066  RMSE=0.265  cRMSE=0.257  corr=0.542
[SMOS] SSS @ fcst4  (composite of 3 cycle(s): ['20260711', '20260723', '20260729']):
  [SSS]  n=8070  bias=-0.074  RMSE=0.232  cRMSE=0.220  corr=0.674
[SMOS] SSS @ fcst5  (composite of 1 cycle(s): ['20260729']):
  [SSS]  n=8070  bias=-0.

## 7. Composite HTML summary

Composite counterpart to 02_validation.ipynb Section 9: gathers every
figure/table this notebook wrote into `COMPOSITE_DIR` into one
self-contained, offline-viewable HTML page, via
`vc.build_html_summary_composite` -- a thin wrapper around the SAME
`val.build_html_summary` the single-cycle notebook uses, so the report
layout/lightbox/grouping code is never duplicated.

In [23]:
html_path = vc.build_html_summary_composite(
    COMPOSITE_DIR, composite_id=COMPOSITE_ID, config=CONFIG,
    cycles=CYCLES,)
print(f"Open {html_path} in a browser to review this composite.")

Open /home/lell/seaforward/forecast/model-runs/Canary_12/validation_composite_bce243d6/validation_cycle_composite_bce243d6.html in a browser to review this composite.


---
## Notes

- **Merging rule**: everything in this notebook merges cycles by **lead
  label**, not calendar date -- `sp1`/`sp2` = model spin-up (first
  `SPINUP_DAYS` days of every cycle), `fcst1`, `fcst2`, ... = forecast
  lead day 1, 2, ... A cycle that doesn't reach a given lead (shorter
  forecast) is simply skipped for that lead, not raised as an error, so
  cycles of different length can be composited together.
  
- **In-situ (Section 8 of 02_validation.ipynb)** are not yet ported to the composite layer -- both
  are natural extensions (in-situ collocation is already
  per-observation/per-platform rather than per-grid-point, so pooling by
  lead time would need the platform's own timestamp reinterpreted as a
  lead relative to its cycle's `CYCLE_DATE`.
